In [2]:
# url = 'https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main'
# resp = requests.get(url)


In [9]:
# import io
# import zipfile
# import requests
# import frontmatter

In [9]:
import io
import zipfile
import requests
import frontmatter


def read_repo_data(repo_owner, repo_name, branch='main', prefix=''):
    """Download a GitHub repo and parse its markdown files.

    Args:
        repo_owner: GitHub user or organization
        repo_name: Repository name
        branch: Branch to download
        prefix: Keep only files whose path starts with this value
                (the zip folder name is removed first)
    """
    url = f'https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/{branch}'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()

    repository_data = []

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for file_info in zf.infolist():
            if file_info.is_dir():
                continue

            # Remove the "faq-main/" folder from the path
            relative_path = file_info.filename.split('/', 1)[1]

            if not relative_path.lower().endswith(('.md', '.mdx')):
                continue
            if not relative_path.startswith(prefix):
                continue

            try:
                with zf.open(file_info) as f_in:
                    content = f_in.read().decode('utf-8', errors='replace')
                data = frontmatter.loads(content).to_dict()
                data['filename'] = relative_path
                repository_data.append(data)
            except Exception as e:
                print(f"Error processing {relative_path}: {e}")

    return repository_data

In [10]:
# faq_data = read_repo_data('DataTalksClub', 'faq', prefix='_questions/')
# print(faq_data[0])
evidently_docs = read_repo_data('evidentlyai', 'docs')
print(evidently_docs[45])


{'title': 'LLM regression testing', 'description': 'How to run regression testing for LLM outputs.', 'content': 'In this tutorial, you will learn how to perform regression testing for LLM outputs.\n\nYou can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.\n\n<Info>\n  **This example uses Evidently Cloud.** You\'ll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>\n\n# Tutorial scope\n\nHere\'s what we\'ll do:\n\n* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.\n\n* **Get new answers**. Imitate generating new answers to the same question.\n\n* **Create and run a Report with Tests**. Compare the answers using LLM-as-a-judge to evalua

In [20]:
import re

text = evidently_docs[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())
print(paragraphs)

# def sliding_window(seq, size, step):
#     if size <= 0 or step <= 0:
#         raise ValueError("size and step must be positive")

#     n = len(seq)
#     result = []
#     for i in range(0, n, step):
#         chunk = seq[i:i+size]
#         result.append({'start': i, 'chunk': chunk})
#         if i + size >= n:
#             break

#     return result

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.

    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)

    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)

    return sections    

evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)
        

# evidently_chunks = []

# for doc in evidently_docs:
#     doc_copy = doc.copy()
#     doc_content = doc_copy.pop('content')
#     # split markdown by level
#     chunks = sliding_window(doc_content, 2000, 1000)
#     for chunk in chunks:
#         chunk.update(doc_copy)
#     evidently_chunks.extend(chunks)

# print(evidently_chunks)

['In this tutorial, you will learn how to perform regression testing for LLM outputs.', 'You can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.', "<Info>\n  **This example uses Evidently Cloud.** You'll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>", '# Tutorial scope', "Here's what we'll do:", '* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.', '* **Get new answers**. Imitate generating new answers to the same question.', '* **Create and run a Report with Tests**. Compare the answers using LLM-as-a-judge to evaluate length, correctness and style consistency.', '* **Build a monitoring Dashboard**. Get plots to track the result